In [ ]:
# Click into this cell and press shift-enter before using this notebook.
# This line loads the ability to use %%ai in your file
# These lines import the Python modules we commonly use in CMPSC 5A
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
path_data="data/"

In [ ]:
# Only run this on Google Colab
# DO NOT RUN on Jupyter Hub !
# It won't do any harm, but it isn't needed and won't work.

!pip install --quiet datascience
!mkdir -p /content/data
path_data = '/content/data'
!gdown --folder https://drive.google.com/drive/folders/1vwShl7JZu0kwjzUVkGOccSZQaI2FZG0b?usp=sharing -O /content/data
!ls /content/data


# Data Cleaning, Part 2, CMPSC 5B, F26


This file contains some further examples of data cleaning in context; this is an excerpt of the notes from CMPSC 5A, F25.

## Background: the `randomSample` function

For data cleaning, it's sometimes helpful to get an initial view of the data by taking a random sample.

In large data sets, if you just look at the first few pages of the data, you may get a distorted idea.   A random sample can sometimes give you a better initial feel for what the range of the data looks like.

Here is a `randomSample` function that takes in two
parameters:
* `myTable` is any `Table` object
* `size` is how many rows we want to sample

In [ ]:
def randomSample(myTable, size):
    n = myTable.num_rows
    allRowIndexes = np.arange(n)
    randomIndexes = \
      np.random.choice(
        allRowIndexes, size, replace=False
      )
    randomRows = myTable.take(randomIndexes)
    return randomRows

Once the function is defined, we can call this function on any table we like.  Here's another table that's in our data directory:

In [ ]:
courseTable = Table.read_table(f"{path_data}/ucsb-s25-courses.csv")
movies = Table.read_table(f"{path_data}/movies_by_year_with_ticket_price.csv")

In [ ]:
randomSample(movies,2)

In [ ]:
randomSample(courseTable,3)

In [ ]:
randomSample(courseTable,5)

A key concept with functions is that we define what parameters we are expecting (in this case, a table, and a number of rows), and then we can use the function to get the result we want for *any* table, and *any* number of rows.

## The `COURSEID` field again

Pulling out a random sample of the `COURSEID` field, we see that it combines three things into one field:


In [ ]:
randomCourses = randomSample(courseTable,5)
courseIds = randomCourses.column("COURSEID")
for courseId in courseIds:
    print(courseId)

Using the `.replace` method for Python Strings, and putting some numbers above to help us count, we can see that:
* `[0:8]` is for the subject area
* `[8:13]` is for course number with two letter suffix
* `[14:]` is the section suffix

In [ ]:

print("          11111111112")
print("012345678901234567890")
for courseId in courseIds:
    print(courseId.replace(" ","@"))

With that, we can write some functions that will turn a COURSEID into its three parts.  We add `.strip` which gets rid of spaces at the beginning and end:

In [ ]:
def courseIdToDept(courseId):
    return courseId[0:8].strip()

def courseIdToCourseNum(courseId):
    return courseId[8:13].strip()

def courseIdToSuffix(courseId):
    return courseId[14:].strip()

We can test these with a for loop and a format string:

To review:
* We put the letter `f` before the opening `"` in the string
* Then we can put variables names in `{}` and they will be substituted in
* Also notice that we can put multiple lines of code inside a for loop; it is run once for each value in the array `courseIds`

In [ ]:
courseIds = randomSample(courseTable,5).column("COURSEID")

for courseId in courseIds:
   dept = courseIdToDept(courseId)
   courseNum = courseIdToCourseNum(courseId)
   suffix = courseIdToSuffix(courseId)
   print(f"courseId={courseId:15} dept={dept:9} courseNum={courseNum:6} suffix={suffix:3}")


## Using functions to make a new table: `apply`

Note: material in this section of the lecture comes from

* [Chapter 8 (Functions and Tables: Intro)](https://inferentialthinking.com/chapters/08/Functions_and_Tables.html)
* [Section 8.1 (Applying a Function to a Column)](https://inferentialthinking.com/chapters/08/1/Applying_a_Function_to_a_Column.html)


Now that we know how to use functions to transform data,
we can use this to:
* make new arrays using the `apply` method.
* add those to a table as columns

Here's what that looks like:

We start with our courseTable:

In [ ]:
courseTable

Remember the three functions from before for converting `COURSEID` values:

In [ ]:
def courseIdToDept(courseId):
    return courseId[0:8].strip()

def courseIdToCourseNum(courseId):
    return courseId[8:13].strip()

def courseIdToSuffix(courseId):
    return courseId[14:].strip()

We can make three new arrays by applying these functions to column 0 of our `courseTable`.  Each result we get back is an array.  The `0` indicates we are applying our functions to column 0.

In [ ]:
departments = courseTable.apply(courseIdToDept, 0)
departments

In [ ]:
courseNums = courseTable.apply(courseIdToCourseNum, 0)
courseNums

In [ ]:
suffixes = courseTable.apply(courseIdToSuffix, 0)
suffixes

Now, we can create a new table by adding these columns to our existing table.  We'll call this `courses`, because this is the basic table that represents all UCSB course data for S25.

Now, we can create a new table by adding these columns to our existing table:

In [ ]:
courses = courseTable.with_columns("dept",departments,"courseNum",courseNums,"suffix",suffixes)

In [ ]:
courses

And with this new tables, we can use the `where` clause to focus our attention on particular departments:

In [ ]:
csCourses = courses.where("dept",are.equal_to("CMPSC"))
pstatCourses = courses.where("dept",are.equal_to("PSTAT"))
artHistory = courses.where("dept",are.equal_to("ARTHI"))

In [ ]:
csCourses

In [ ]:
pstatCourses

In [ ]:
artHistory

## Review: Bar Graphs

Let's now return to some material from:

* [Chapter 7 (Visualization)](https://inferentialthinking.com/chapters/07/Visualization.html)

Let's see if we can make a bar graph of the data from `courseTable`.

But let's work with the `newTable` that has `dept`, `courseNum`, `suffix`.  And let's cut the data down to just three departments: `CMPSC`, `MATH`, `PSTAT`, just to make it a bit easier to work with.

It will be really handy to refer to this documentation:
* <https://www.data8.org/datascience/reference-nb/datascience-reference.html#Table.where-Predicates>

In this context, a `predicate` is something that's true or false about rows of our data.  The one we will use here is `are.contained_in` which works with an array.

In [ ]:
desired_departments = make_array("CMPSC", "PSTAT", "MATH")

In [ ]:
coursesSubset = newTable.where("dept", are.contained_in(desired_departments))

In [ ]:
coursesSubset.num_rows

Still pretty big, but a lot smaller than what we had before.

The first step is to group our data.

## Review: Grouping Data

In [ ]:
allCoursesGroupedByDepartment = newTable.group("dept")
selectedCoursesGroupedByDepartment = coursesSubset.group("dept")

In [ ]:
selectedCoursesGroupedByDepartment

In [ ]:
allCoursesGroupedByDepartment

What we get back is a new table that shows the count of how many rows were in the original data set with each of the distinct values in the original.

# Simple Bar Graphs

Let's look again at what happens when we group the courses from three departments by subject area:

In [ ]:
desired_departments = make_array("CMPSC", "PSTAT", "MATH")
coursesSubset = newTable.where("dept", are.contained_in(desired_departments))
coursesSubset.group("dept")

When we have a nice clean table like this with just two columns:
* A label
* A number

We can get a bar chart very easily with this:

In [ ]:
coursesSubset.group("dept").barh("dept","count")

But, as we'll see, this table comparing the count of courses between PSTAT, MATH and CMPSC for S25 is not entirely accurate.

For one thing, the data has multiple rows for each course: one for the lecture, and another for the section.

So, while this is an interesting graph, we need to do some more work before we can rely on this for decision making.

## Data Cleaning: A first look

Data Cleaning typically involves removing data that:
* Contains errors
* Is inconsistent
* Accurately reflects the question we are asking.

As an example, suppose we want to know which departments (or more precisely, which subject areas) offer the most undergraduate courses at UCSB?

Here's a quick and dirty analysis: we can group the courses table by department, and then sort, descending by the count:


In [ ]:
courses.group("dept").sort("count",descending=True).show(5)

By this count, the top five departments are CHEM, PSTAT, PSY, PHYS and MCDB.

But, is this count accurate?

As we'll see if we look into the data in more detail, there are a few things throwing this off.

1. We haven't distinguished between grad and undergrad courses yet.
2. We are counting both lectures and sections.
3. Some departments offer a large number of "independent study" and "research seminar" courses with low enrollments that, while they are an important part of the UCSB experience, are in a different category, and may distort the numbers.  Those should perhaps be counted separately.


Let's look at each of these problems separately.  First, how can we check whether this table contains both grad and undergrad courses?

One way is to exploit the fact that the COURSEID field has particular digits in specific columns of the string.   We know that COURSEID has the format:
```
SSSSSSSSNNNXX-AA
```
For example, here's the course number for this course, CMPSC 5A, in this format,
along with column numbers starting from 0 and going up to 15

```
          111111
0123456789012345
SSSSSSSSNNNXX-AA
CMPSC     5A -1
```

where:
* `SSSSSSSS` is the subject area
* `NNN` is the course number
* `XX` is the course number suffix
* `AA` is the number of the lecture section

Note that:
* If a course is a lower division undergrad course, `COURSEID[8]` will be blank (i.e. a space character in Python ` `).
* If a course is an upper division undergrad course, `COURSEID[8]` will be the `1` character.
* Anything else (e.g. `2`, `5`, etc.) is a graduate course.

Suppose we write a function that takes a courseid as a parameter, and then checks whether the character in position `[8]` is a `' '` (space) or a `'1'`:


In [ ]:
def is_undergrad(courseid):
    """ returns true if course is undergrad, otherwise returns false """
    return courseid[8]==' ' or courseid[8]=='1'

Once we have this method, we can use it like this:

In [ ]:
courses.where("COURSEID", is_undergrad)

But why does that work?  
* Short version: because `is_undergrad` is a special type of function called a *predicate*
* To understand more, let's look at this `.where()` method more closely.

## Short digression: the `table.where(...)` method

We've seen the `table.where(...)` method before.  It's one of the two ways we can slice a table horizontally:
* `table.take(...)` is used when we want to specify table rows by their *index*, i.e. the row number (starting with `0`).  
* `table.where(...)` is used when we want to specify table rows by *criteria*, i.e. something that is true about those rows, and not true of the other rows.

An example of `.where(...)` is when we want all courses with more than 99 students:


In [ ]:
courses.where("ENROLLED", are.above(99))

How can we remember all of the things that come after `.are`?  For example, how can we know whether it should be:

* `courses.where("ENROLLED", are.greater_than(99))`
* `courses.where("ENROLLED", are.larger_than(99))`
* `courses.where("ENROLLED", are.above(99))`
* `courses.where("ENROLLED", are.bigger_than(99))`

All of those are *plausible* but only one of those is correct.  How can we know?

Please know: Your instructor isn't planning to memorize them, and doesn't expect you to either.  Here's how you can look them up:

Here are four ways:
* The most convenient way is to put `help(are)` at a Python prompt (e.g. in a Jupyter Notebook in a code cell).  We'll show what that looks like below.
* The Canvas site links to the documentation for the `datascience` module.  We can look it up, but we need to know what we are looking for.  These things that go after `are` are callled *where predicates*. We'll talk more about *predicates* below.
* On a PrairieLearn quiz, if you need where predicates, we'll typically provide a "Hint" dropdown with a reference for the predicates.
* You can try using ai, but beware that you will need to specify that you are using the "Python datascience module" and even then, the answers may or may not be accurate, so always check them out before blindly trusting the ai.

### What `help(are)` looks like:

You can often get helpful documentation about things in Python by passing the name as a parameter to the `help(...)`
function, as shown below.

After you've found what you need, it's a good idea to do one of these so the output doesn't clutter up the notebook:
* delete the cell
* comment it out (put a `#` in front of it (pound sign, octothorpe, hash tag))
* use `Edit / Clear Cell Output` from the Jupyter Notebook menu

Note that in the output, these called *predicates*.

In [ ]:
help(are)

## A short digression about *predicates*

Function definitions that always return `True` or `False` (boolean values) have a special name: they are called *predicates*.   

The reason it's important to learn this special word *predicate* is that, as we've seen, it shows up in the documentation for the `where` method that we use on tables.

We've seen before that we can select rows out of a table with a `.where(...)` method such as this.



In [ ]:
courses.where("ENROLLED", are.above(99))

So, here, `are.above(99)` is a predicate.  What's going on here is a little sutble: `are.above(99)` returns a function that takes a value as a parameter, and determines whether it is `>99`.  
* If so, it returns `True`
* If not, it returns `False`

Let's explore that by giving `are.above(99)` a name using an assignment statement, and then using it
as a function call:

In [ ]:
above99 = are.above(99)
above99(100)

In [ ]:
above99(90)

We can see that `are.above(99)` can be treated as a function that takes a value as it's parameter, and returns either `True` or `False`.

So when we write:
```
courses.where("ENROLLED", are.above(99))
```

what is happening is this:

```
for every row in the table:
   we pass the value of "ENROLLED" in that row to "are.above(99)"
   if we get back True, we include that row in the result
   if we get back False, we do NOT include that row in the result
```
  
Let's see what that looks like:

In [ ]:
courses.where("ENROLLED", are.above(99))

## Using our *own* predicate functions in a `.where` call:


In the same way, we can use our own predicate function:
```
def is_undergrad(courseid):
    """ returns true if course is undergrad, otherwise returns false """
    return courseid[8]==' ' or courseid[8]=='1'
```
in a where clause like this:

```
courses.where("ENROLLED", is_undergrad)
```

What is happening is this:

```
for every row in the table:
   we pass the value of "ENROLLED" in that row to `is_undergrad(...)` as it's parameter.
   if we get back True, we include that row in the result
   if we get back False, we do NOT include that row in the result
```
Let's see what that looks like:


In [ ]:
def is_undergrad(courseid):
    """ returns true if course is undergrad, otherwise returns false """
    return courseid[8]==' ' or courseid[8]=='1'

In [ ]:
undergrad_courses = courses.where("COURSEID",is_undergrad)
undergrad_courses

It may be difficult to know whether that result is giving us what we want.  

So let's try a few things to see.

First let's check the counts:

In [ ]:
print(f"courses.num_rows = {courses.num_rows}")
print(f"undergrad_courses.num_rows = {undergrad_courses.num_rows}")

Let's also randomly sample.  What we would hope to see is that:
* in our random sample of the full `courses` table, we sometimes get grad courses.
* with our random sample from `undergrad_courses` we never do.


In [ ]:
randomSample(courses, 5)

In [ ]:
randomSample(undergrad_courses,5)

Ok, that looks good so far.  But here's a way to get even more confidence. Let's make a predicate function that takes a `COURSEID` value and tells us whether we have a graduate course or not:

In [ ]:
def isGraduate(courseNum):
    """ True if courseNum isn't a space and it isn't a 1 (so it must be a graduate course) """
    return courseNum[8] != ' ' and courseNum[8] != '1'

Now we can pull out all of the graduate courses:


In [ ]:
graduate_courses = courses.where("COURSEID", isGraduate)

In [ ]:
print(f"courses.num_rows = {courses.num_rows}")
print(f"undergrad_courses.num_rows = {undergrad_courses.num_rows}")
print(f"graduate_courses.num_rows = {graduate_courses.num_rows}")
print(f"undergrad_courses.num_rows + graduate_courses.num_rows = {undergrad_courses.num_rows + graduate_courses.num_rows}")

This gives us much more confidence that our functions are calculating correctly!

We can also check random rows from our `graduate_courses` table:

In [ ]:
randomSample(graduate_courses, 5)

## The important of testing

When we write our own functions, it's very important to test them to make sure they do
what we want them to do.   When developing this lecture, I first wrote this version
of the function to check for graduate courses.   Can you spot the bug?

I didn't at first!   (After 5 minutes, I saw it, but only because when I tested, I was getting incorrect results.)

In [ ]:
# This version has a bug
def isGraduate_buggy(courseId):
    """ True if courseNum isn't a space and it isn't a 1 (so it must be a graduate course) """
    courseId[8] != ' ' and courseId[8] != '1'

Here's what happens when we try to use it:

In [ ]:
graduate_courses_take2 = courses.where("COURSEID",isGraduate_buggy)
graduate_courses_take2

In [ ]:
graduate_courses_take2.num_rows

Do you see the problem?

## Answering a question about course

To review, suppose we want to answer these questions about UCSB undergrad courses in Spring 2025:
* Which subject areas offered the most courses?
* Which subject areas offered the fewest courses?

We now know that to answer this with the data we have, we need to:
* Exclude graduate courses
* Count only lecture sections (i.e. don't  count discussion sections as separate courses)

We may also want to define a minimum enrollment for a course to count; let's say 10 students.

In addition to our `isUndergrad` predicate, let's define two more:

In [ ]:
def is_lecture(sectionNum):
    # if section number is divisible by 100, it's a lecture
    return sectionNum % 100 == 0

In [ ]:
is_lecture(100)

In [ ]:
is_lecture(101)

In [ ]:
def tenOrMoreStudents(enrollment):
    return enrollment > 10

Now we can apply all of these predicates to our dataset:

In [ ]:
coursesToCount = courses \
    .where("COURSEID", is_undergrad) \
    .where("SECTION", is_lecture) \
    .where("ENROLLED", tenOrMoreStudents)

In [ ]:
coursesToCount

In [ ]:
Now, can group by the `dept` field (really "subject area"):

In [ ]:
coursesToCount.group("dept")

And, we can sort to see the top 10 and bottom 10 in this table:

In [ ]:
coursesToCount.group("dept").sort("count", descending="True")

We can see that if we want the top five, there's actually a problem, since there is a tie!
So we'll use the trick we've used before: find the value in row(4) (the fifth value in the table), then show all rows with a count >= to that value:

In [ ]:
fifthHighest = coursesToCount.group("dept").sort("count", descending="True").column("count").take(4)
fifthHighest

In [ ]:
coursesToCount.group("dept").where("count", are.above_or_equal_to(fifthHighest)).sort("count",descending="True")

In [ ]:
For the bottom five, let's first look at the bottom ten:

In [ ]:
coursesToCount.group("dept").sort("count")

We can see that there is *at least* a ten-way tie for last place, with many subject areas having only one course.  So we'll need to just list all of the subject areas with only one course:

In [ ]:
coursesToCount.group("dept").where("count",are.equal_to(1)).show()

That's probably a bit more like what we want.  What if we wanted to know the total enrollment?  Can we sum up the `ENROLLED` field for each group?  Yes we can!

# Using `group` for things other than counting

The most basic use of the `group` function is to group rows in a table based on a particular column,
and then count up the values.

However, we can take it a step further using functions such as `sum`, `min`, `max`, and others; basically any function that can take a `numpy.ndarray` as it's argument.

First, we'll show what this looks like.  Then we'll dive in a bit deeper.

Here's: `coursesToCount.group("dept",sum)`


In [ ]:
coursesToCount.group("dept",sum)

What we get back is the 'sum' function applied to each column of data.
* The numbers in `ENROLLED sum` show the total of the enrollment
* The numbers in `MAXRENROLL sum` show the total of the max enrollment, i.e. how big the class could be if full
* The numbers if `QUARTER sum` and `SECTION sum` are not really meaningful; they are adding up numbers that are really just identifiers.  We might have done well to exclude those columns before doing our grouping.
* Lecture sum shows us the number of rows for which that field is True.  Since that's every row of data, this actually shows us the same as the counts from the original `.group()` call.

So really, a better way to use `group` with the `sum` function is to first use `select` so that we only have columns on which
finding a sum is meaningful.  
    
For example, consider this table:

In [ ]:
coursesToCount.select("COURSEID", "dept","ENROLLED","MAXENROLL")

If we take this table, and group by dept and take the sum, we get this:

In [ ]:
coursesToCount.select("COURSEID", "dept","ENROLLED","MAXENROLL").group("dept",sum)

You can see that the COURSEID sum column is blank, because it doesn't make any sense to find the "sum" of the courseid values.   So, in the next example, we'll omit it.  Here is how we can find the subject areas with the most students enrolled.

In [ ]:
coursesToCount \
    .select("COURSEID", "dept","ENROLLED","MAXENROLL") \
    .group("dept",sum).sort("ENROLLED sum",descending=True)

We can also use the `max` function to determine which subject area has a single class with the largest enrollment.  This is applying the `max` function across all of the ENROLLED values in each group by subject area.

In [ ]:
coursesToCount \
    .select("dept","ENROLLED","MAXENROLL") \
    .group("dept",max).sort("ENROLLED max",descending=True)

Let's give this table a name with an assignment statement

In [ ]:
largestCourseSizes = coursesToCount \
    .select("dept","ENROLLED","MAXENROLL") \
    .group("dept",max).sort("ENROLLED max",descending=True)

In [ ]:
largestCourseSizes

Suppose we wanted to add to this table the actual name of the course that is the largest one in each subject area.

First, here's an *incorrect* answer:

In [ ]:
largestCourseSizes = coursesToCount \
    .select("COURSEID", "dept","ENROLLED","MAXENROLL") \
    .group("dept",max).sort("ENROLLED max",descending=True)
largestCourseSizes

We might think this is saying that ECON 199RA-2 is the largest class offered by ECON. But that is not the case.

Instead, `ECON 199RA-2` is the `max` of the `COURSEID` column *lexigographically*, i.e. the one that would occur last if we sorted by this column!  That's not really meaningful if we are looking to identify the largest course (by enrollment) taught by ECON.

Instead, we can write a function that takes the `dept` as a parameter, and returns as it's value, the course id of the largest course in that department.  Then we can use `apply` to make a new `numpy.ndarray` of those values, and then we can add that column to the table.

Here's how that looks. First let's do it by hand for just `ECON`


Here are all of the ECON courses, in descending order by enrollment:

In [ ]:
coursesToCount.where("dept", are.equal_to("ECON")).sort("ENROLLED",descending=True).show(5)

In [ ]:
What we want is the `COURSEID` field from the top row.  So that's this:


In [ ]:
coursesToCount.where("dept", are.equal_to("ECON")).sort("ENROLLED",descending=True).column('COURSEID').item(0)

**This doesn't work if there are ties**.  Note that if there are ties for the largest course, this method won't work; we'll only pull out one of the courses that has that many students.  Maybe we'll come back later and show how we'd solve the problem if there were ties.  But it's a good start!

Let's turn this into a function.  Instead of hard coding `"ECON"`, we replace this with a parameter.

In [ ]:
def deptToLargestCourse(dept):
    return coursesToCount.where("dept", are.equal_to(dept)).sort("ENROLLED",descending=True).column('COURSEID').item(0)

Now lets test this on a few departments:

In [ ]:
deptToLargestCourse("ECON")

In [ ]:
deptToLargestCourse("CMPSC")

In [ ]:
deptToLargestCourse("CLASS")

Seems to be working.  Let's try the apply function now.

In [ ]:
largestCourseSizes = coursesToCount \
    .select("dept","ENROLLED") \
    .group("dept",max).sort("ENROLLED max",descending=True) \
    .with_column("Largest Enrollment Course", largestCourseSizes.apply(deptToLargestCourse, "dept"))
largestCourseSizes.show()

## Bar Graphs

We can now make some bar graphs with this data.

Here's a bar graph of the number of courses offered by each department:


In [ ]:
counts = coursesToCount.group("dept")
counts

In [ ]:
counts.barh("dept") # Bar graph of count, by department

Or we can make a bar graph by enrollment:

In [ ]:
enrollment = coursesToCount.select("dept","ENROLLED").group("dept",sum).sort("ENROLLED sum",descending=True)
enrollment.show(10)

In [ ]:
enrollment.barh("dept","ENROLLED sum")